# CALENDAR: Análisis Exploratorio de Datos y Parametrización de la Matriz de Disponibilidad Temporal

El archivo `calendar.txt` constituye la base estructural cronológica de la EMT de Madrid, parametrizando la disponibilidad operativa y la recurrencia semanal de la oferta de transporte de superficie a lo largo del año. En la arquitectura de este proyecto de Machine Learning, el análisis de este conjunto de datos representa un pilar crítico de producción: actúa como el indexador temporal que define el régimen de marcha de la flota y establece la segregación binaria de los vectores de servicio entre jornadas laborables regulares, operaciones de fin de semana y ventanas estacionales de conectividad.

### Objetivos del Diagnóstico:
1. **Auditoría de Vigencia y Cobertura:** Implementar controles defensivos de integridad sobre los horizontes cronológicos (`start_date`, `end_date`) para verificar la continuidad ininterrumpida de las ventanas de servicio y neutralizar solapamientos temporales corruptos.

2. **Discretización Cíclica de la Oferta:** Validar la consistencia de las banderas booleanas de los días de la semana para asegurar una correcta modelización de patrones pendulares y estructurar las bases de la ingeniería de características horaria.

3. **Saneamiento Relacional de Claves:** Validar la unicidad y consistencia del identificador de servicio (`service_id`) para garantizar un engranaje y linaje relacional perfecto con las tablas operativas de control de viajes en fases posteriores del pipeline.

4. **Optimización del Linaje Físico:** Reestructurar el esquema de memoria del catálogo mediante tipos de datos compactos y ligeros de PyArrow (`int8` para banderas booleanas y `timestamp` para fechas), mitigando el consumo de recursos y acelerando las operaciones de fusión relacional.

Se inicia el flujo de ingeniería de datos realizando una inspección estructural preliminar para auditar la dimensionalidad y consistencia de los vectores de entrada del DataFrame `calendar`.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print("====== INICIANDO CARGA OPTIMIZADA DE PATRONES DE CALENDARIO (PYARROW SCHEMA) ======\n")

# 1. Especificación del esquema físico estricto mediante el motor de PyArrow para mitigar la huella de RAM
# Se comprimen las banderas booleanas de los días de la semana y fechas a enteros de 8 bits (int8)
calendar_dtypes = {
    "service_id": "string[pyarrow]",
    "monday": "int8[pyarrow]",
    "tuesday": "int8[pyarrow]",
    "wednesday": "int8[pyarrow]",
    "thursday": "int8[pyarrow]",
    "friday": "int8[pyarrow]",
    "saturday": "int8[pyarrow]",
    "sunday": "int8[pyarrow]",
    "start_date": "int32[pyarrow]",
    "end_date": "int32[pyarrow]"
}

# 2. Carga del archivo original aplicando el mapa de tipos controlado
calendar = pd.read_csv('../data/raw/gtfs_eda_snapshot/calendar.txt', encoding='utf-8', dtype=calendar_dtypes)
calendar['service_id'] = calendar['service_id'].str.strip().str.upper().astype("string[pyarrow]")

print("====== ESTRUCTURA ESTRUCTURAL INICIAL DE CALENDAR (METADATOS) ======\n")
print(calendar.info())

print("\nMuestra forense preliminar de registros originales (Catálogo Completo):")
display(calendar.head(5))


====== INICIANDO CARGA OPTIMIZADA DE PATRONES DE CALENDARIO (PYARROW SCHEMA) ======

====== ESTRUCTURA ESTRUCTURAL INICIAL DE CALENDAR (METADATOS) ======

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   service_id  5 non-null      string        
 1   monday      5 non-null      int8[pyarrow] 
 2   tuesday     5 non-null      int8[pyarrow] 
 3   wednesday   5 non-null      int8[pyarrow] 
 4   thursday    5 non-null      int8[pyarrow] 
 5   friday      5 non-null      int8[pyarrow] 
 6   saturday    5 non-null      int8[pyarrow] 
 7   sunday      5 non-null      int8[pyarrow] 
 8   start_date  5 non-null      int32[pyarrow]
 9   end_date    5 non-null      int32[pyarrow]
dtypes: int32[pyarrow](2), int8[pyarrow](7), string(1)
memory usage: 258.0 bytes
None

Muestra forense preliminar de registros originales (Catálogo Completo):


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,LA,1,1,1,1,1,0,0,20261120,20271231
1,SA,0,0,0,0,0,1,0,20261120,20271231
2,FE,0,0,0,0,0,0,1,20261120,20271231
3,LJ,1,1,1,1,0,0,0,20261120,20271231
4,VV,0,0,0,0,1,0,0,20261120,20271231


### Diagnóstico de Dimensionalidad y Estrategia de Preservación de Datos

La inspección estructural preliminar a través de la salida del método `.info()` revela una matriz de datos sumamente compacta y de alta densidad de información, compuesta estrictamente por **5 registros únicos y 10 columnas vectoriales**. A diferencia de los datasets macroscópicos del estándar GTFS (como el catálogo de nodos de trazado o el histórico de paradas), `calendar.txt` opera como una tabla maestra de constantes operativas semanales, exhibiendo una completitud perfecta con un índice de registros nulos igual a cero (`5 non-null` en todas las variables).

**Criterio Técnico de Negocio e Impacto en el Linaje:**
Dada la naturaleza del proyecto, orientada a modelar y predecir la puntualidad de las líneas de la EMT de Madrid, la conservación absoluta de la dimensionalidad de esta tabla es una decisión estratégica obligatoria. Las claves alfabéticas del identificador (`service_id`) segmentan la red en perfiles de demanda independientes, las siete banderas booleanas de los días de la semana parametrizan los ciclos diarios de congestión y los límites cronológicos (`start_date`, `end_date`) confinan la vigencia legal del servicio. Para maximizar la eficiencia computacional, el esquema se consolida desde la carga en disco bajo los tipos de datos optimizados de PyArrow (`string`, e `int8` e `int32` para las fechas), comprimiendo la huella de memoria RAM al mínimo absoluto. Conservar íntegra esta estructura binaria sin realizar reducciones es indispensable para que los modelos de Machine Learning puedan heredar en fases posteriores el contexto del comportamiento del tráfico madrileño (como las asimetrías extremas entre jornadas laborables regulares, operaciones de fin de semana o la transición de vísperas festivas), protegiendo la varianza temporal del sistema.

### 📘 Diccionario de datos

Este notebook utiliza el diccionario de datos definido para el conjunto `calendar`.  

El documento completo se encuentra en:

👉 [`../docs/04_calendar_Dict.md`](../docs/04_calendar_Dict.md)

| Columna | Tipo de Dato (Pandas) | Descripción de Negocio | Rol en el Modelo / Importancia |
| :--- | :--- | :--- | :--- |
| **`service_id`** | `String` a `string[pyarrow]` | Identificador alfabético mnemotécnico único que parametriza un patrón operativo semanal específico de la EMT (e.g., `LA`: Laborables, `SA`: Sábados, `FE`: Festivos). | **Clave Primaria**. Indexador principal de disponibilidad temporal; vincula relacionalmente los regímenes de marcha semanales con la tabla operativa de viajes `trips`. |
| **`monday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los lunes. | **Bandera Booleana**. Predictor de ciclo semanal; clave para aislar la fricción y asimetría de la demanda en el arranque de la semana laborable. |
| **`tuesday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los martes. | **Bandera Booleana**. Predictor de ciclo semanal; mapea la regularidad del tráfico en las ventanas ordinarias laborables. |
| **`wednesday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los miércoles. | **Bandera Booleana**. Predictor de ciclo semanal; mapea la regularidad del tráfico en las ventanas ordinarias laborables. |
| **`thursday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los jueves. | **Bandera Booleana**. Predictor de ciclo semanal; mapea la regularidad del tráfico en las ventanas ordinarias laborables. |
| **`friday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los viernes. | **Bandera Booleana**. Predictor de ciclo semanal; factor crítico para modelar el comportamiento de vísperas y la transición hacia el fin de semana. |
| **`saturday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los sábados. | **Bandera Booleana**. Predictor de ciclo semanal; aísla escenarios de baja fricción vehicular por la caída masiva de la demanda laboral. |
| **`sunday`** | `Int64` a `int8[pyarrow]` | Indicador binario (`1` o `0`) que determina si el régimen de servicio codificado se encuentra activo o inactivo durante las jornadas de los domingos. | **Bandera Booleana**. Predictor de ciclo semanal; aísla escenarios de baja fricción vehicular por la caída masiva de la demanda laboral. |
| **`start_date`** | `Int32` a `timestamp[us][pyarrow]` | Fecha oficial que marca el inicio de la vigencia del patrón de calendario, cargada como entero comprimido (`AAAAMMDD`) y transformada a objeto temporal durante el pipeline. | **Descriptor Cronológico**. Variable temporal raíz; delimita la validez de la oferta y actúa como la base para la inminente ingeniería de fechas. |
| **`end_date`** | `Int32` a `timestamp[us][pyarrow]` | Fecha oficial que marca la finalización de la vigencia del patrón de calendario, cargada como entero comprimido (`AAAAMMDD`) y transformada a objeto temporal durante el pipeline. | **Descriptor Cronológico**. Variable temporal raíz; delimita la validez de la oferta y actúa como la base para la inminente ingeniería de fechas. |



### Control de Duplicados e Integridad de Claves Primarias

Antes de proceder con el análisis estacional de las fechas, es indispensable auditar el catálogo cronológico para certificar la ausencia de redundancias estructurales o conflictos de unicidad en sus identificadores. Al operar `calendar.txt` como la tabla maestra que parametriza las constantes operativas semanales de la EMT de Madrid, se implementa un control de calidad bidimensional estricto:

1. **Redundancia Absoluta (Filas Idénticas):** Evaluación de registros completamente duplicados en todas sus columnas vectoriales, anomalía habitualmente provocada por fallos en la exportación o persistencia del sistema de origen. Su saneamiento es directo mediante filtrado.

2. **Colisiones de Clave Primaria (`service_id`):** Diagnóstico forense sobre la unicidad del identificador de servicio. Se buscan anomalías donde una misma clave alfabética (e.g., 'LA' o 'SA') registre dos patrones de días o ventanas de vigencia distintas. Desde la perspectiva de negocio y modelado, este escenario es crítico: una colisión de clave rompería el determinismo relacional del pipeline, provocando cruces ambiguos en las tablas transaccionales de control de viajes que corromperían las predicciones de puntualidad en el resto del ecosistema de datos.

El objetivo de esta fase es cuantificar la presencia de ambos fenómenos para validar la salud estructural del catálogo temporal antes de transferir las variables hacia los módulos de validación y limpieza de producción.

In [3]:
print("====== REPORTE DE CONTROL DE DUPLICADOS EN CALENDARIO (CALENDAR) ======\n")

# 1. Cuantificación de redundancias absolutas y colisiones de clave primaria
duplicados_totales = calendar.duplicated().sum()
duplicados_servicio = calendar['service_id'].duplicated().sum()

print(f"-> Registros 100% idénticos en bruto detectados: {duplicados_totales}")
print(f"-> Colisiones en identificador 'service_id' encontradas: {duplicados_servicio}")

# 2. Cláusulas de control y toma de decisiones operativas de backend
if duplicados_totales > 0:
    print("\nMuestra de filas de calendario 100% idénticas repetidas:")
    df_duplicados_completos = calendar[calendar.duplicated(keep=False)]
    display(df_duplicados_completos.sort_values(by=['service_id']))
    print("\n[RESOLUCIÓN BACKEND]: Aplicación defensiva de '.drop_duplicates()' en el flujo de producción.")

if duplicados_servicio > 0:
    print("\n¡ALERTA CRÍTICA! Se han identificado claves primarias duplicadas en la matriz:")
    df_duplicados_servicio = calendar[calendar.duplicated(subset=['service_id'], keep=False)]
    display(df_duplicados_servicio.sort_values(by=['service_id']))
    print("\n[RESOLUCIÓN BACKEND]: Desvío forzado inmediato hacia el módulo de cuarentena en 'src/limpieza.py'.")
else:
    print("\nTodo limpio: Cada 'service_id' parametriza un régimen de marcha semanal único y determinista.")

====== REPORTE DE CONTROL DE DUPLICADOS EN CALENDARIO (CALENDAR) ======

-> Registros 100% idénticos en bruto detectados: 0
-> Colisiones en identificador 'service_id' encontradas: 0

Todo limpio: Cada 'service_id' parametriza un régimen de marcha semanal único y determinista.


### Diagnóstico de Completitud y Gestión de Valores Ausentes

Una vez validada la unicidad de las claves primarias, se procede a realizar un diagnóstico sistemático sobre la integridad y completitud de las variables del catálogo temporal. Se cuantifica la presencia de registros nulos o valores faltantes (`NaN / None`) desglosados por columna para evaluar la salud interna de la matriz de disponibilidad de la red.

Desde la perspectiva del análisis predictivo y de negocio, el control de nulos en `calendar.txt` es de severidad absoluta: cualquier ausencia de información en variables raíz como las banderas de los días de la semana o los límites cronológicos de vigencia (`start_date`, `end_date`) constituye una anomalía incorregible de origen. Un nulo en estos campos inhabilitaría el engranaje relacional del pipeline, impidiendo asociar de forma determinista los viajes con su correspondiente patrón de tráfico diario. El objetivo de este bloque es certificar empíricamente la completitud total del archivo antes de proceder con el parseo y la conversión de las variables hacia objetos de fecha nativos.

In [4]:
print("====== REPORTE DE CONTROL DE VALORES AUSENTES (CALENDAR) ======\n")

# 1. Cuantificación de registros ausentes por columna mediante evaluación de nulidad vectorial
nulos_por_columna = calendar.isna().sum()
total_nulos_dataframe = calendar.isna().any(axis=1).sum()

print("Inventario de registros nulos por columna:")
print(nulos_por_columna)

# 2. Evaluación de registros incompletos para la toma de acciones correctivas
if total_nulos_dataframe > 0:
    print(f"\n¡Atención! Se han detectado {total_nulos_dataframe} registros incompletos en la matriz:")
    df_nulos = calendar[calendar.isna().any(axis=1)]
    display(df_nulos)
    print("\n[RESOLUCIÓN BACKEND]: Los registros incompletos corrompen las reglas horarias. "
          "Se programará su desvío forzado inmediato hacia el módulo de cuarentena en 'src/limpieza.py'.")
else:
    print("\nTodo limpio: El 100% de la matriz cronológica de patrones semanales está completa (cero nulos).")

====== REPORTE DE CONTROL DE VALORES AUSENTES (CALENDAR) ======

Inventario de registros nulos por columna:
service_id    0
monday        0
tuesday       0
wednesday     0
thursday      0
friday        0
saturday      0
sunday        0
start_date    0
end_date      0
dtype: int64

Todo limpio: El 100% de la matriz cronológica de patrones semanales está completa (cero nulos).


### Ingeniería de Características: Parseo y Validación de Horizontes Cronológicos

Una vez garantizada la completitud perfecta de la matriz de calendario, se procede a aplicar técnicas de ingeniería de características (*Feature Engineering*) orientadas a la normalización de los tipos de datos temporales. El estándar GTFS codifica originalmente las variables de vigencia `start_date` y `end_date` en un formato numérico comprimido (`int32` con estructura `AAAAMMDD`), una representación que resulta inoperable para realizar cálculos de deltas o segmentaciones estacionales.

Se implementa un algoritmo de transformación rígido para parsear estos campos hacia objetos temporales nativos forzando el tipo de marca de tiempo ligero de PyArrow (`timestamp[us][pyarrow]`). Una vez realizada la conversión, se ejecuta una validación de consistencia cronológica basada en leyes de negocio: se auditada de forma estricta que la fecha de finalización oficial del servicio (`end_date`) sea estrictamente mayor o igual a la fecha de inicio (`start_date`) dentro de un mismo identificador. Cualquier registro que viole esta condición secuencial temporal se catalogará como una anomalía cronológica insalvable (retroceso temporal) y se derivará de forma forzada hacia el módulo de cuarentena de producción.

In [5]:
print("====== INICIANDO CONVERSIÓN Y AUDITORÍA CRONOLÓGICA DE HORIZONTES ======\n")

# 1. Parseo vectorial de formato comprimido forzando el tipo temporal nativo de PyArrow
calendar['start_date'] = pd.to_datetime(calendar['start_date'].astype(str), format='%Y%m%d').astype("timestamp[us][pyarrow]")
calendar['end_date'] = pd.to_datetime(calendar['end_date'].astype(str), format='%Y%m%d').astype("timestamp[us][pyarrow]")

print("-> Tipos de datos cronológicos normalizados con éxito.")
print(f"   Variable 'start_date': {calendar['start_date'].dtype}")
print(f"   Variable 'end_date':   {calendar['end_date'].dtype}\n")

# 2. Control de consistencia cronológica mediante evaluación de gradiente secuencial
indices_cronologia_inversa = calendar[calendar['end_date'] < calendar['start_date']].index
total_inversiones = len(indices_cronologia_inversa)

print(f"-> Registros con incoherencia cronológica (end_date < start_date): {total_inversiones}")

# 3. Aislamiento y toma de decisiones operativas de backend
if total_inversiones > 0:
    print("\n¡ALERTA CRÍTICA! Se han identificado horizontes de vigencia temporales invertidos:")
    df_cronologia_corrupta = calendar.loc[indices_cronologia_inversa]
    display(df_cronologia_corrupta)
    print("\n[RESOLUCIÓN BACKEND]: Los viajes en el tiempo corrompen el motor de asignación. "
          "Estos registros se desviarán para su aislamiento hacia el archivo de cuarentena.")
else:
    print("\nTodo limpio: El 100% de los horizontes de vigencia cumplen las leyes de consistencia cronológica.")

====== INICIANDO CONVERSIÓN Y AUDITORÍA CRONOLÓGICA DE HORIZONTES ======

-> Tipos de datos cronológicos normalizados con éxito.
   Variable 'start_date': timestamp[us][pyarrow]
   Variable 'end_date':   timestamp[us][pyarrow]

-> Registros con incoherencia cronológica (end_date < start_date): 0

Todo limpio: El 100% de los horizontes de vigencia cumplen las leyes de consistencia cronológica.


### Caracterización Estadística y Auditoría Dimensional del Calendario

Una vez consolidadas las fases de depuración estructural, tipado cronológico binario (`timestamp[us][pyarrow]`) y consistencia secuencial de horizontes, se procede a realizar la auditoría dimensional de cierre sobre la matriz de disponibilidad temporal. Al tratarse `calendar.txt` de un catálogo maestro compacto de alta densidad constituido estrictamente por cinco registros únicos, la mejor práctica metodológica consiste en realizar una auditoría tabular exhaustiva sobre la totalidad del DataFrame.

Este diagnóstico descriptivo final permite verificar simultáneamente los horizontes macro-estacionales de la red de la EMT de Madrid y validar de forma agregada que las banderas booleanas de los ciclos semanales respondan fielmente a las reglas de negocio de la flota antes de transferir el conjunto de datos consolidado hacia los módulos de visualización y el modelo predictivo global.

In [6]:
print("====== CARACTERIZACIÓN ESTADÍSTICA Y AUDITORÍA DE HORIZONTES (CALENDAR) ======\n")

# 1. Extracción forense de los límites macro-estacionales absolutos de la red de la EMT
# Se utiliza .dt.strftime de forma vectorizada sobre los objetos timestamp de PyArrow
fecha_minima_real = calendar['start_date'].min().strftime('%Y-%m-%d')
fecha_maxima_real = calendar['end_date'].max().strftime('%Y-%m-%d')

print(f"-> Horizonte Cronológico Apertura de Oferta: {fecha_minima_real}")
print(f"-> Horizonte Cronológico Cierre de Oferta:   {fecha_maxima_real}\n")
print("-> Matriz Completa de Disponibilidad Temporal y Banderas Semanales de Negocio:")

# 2. Copia defensiva y conversión a tipos estándar para el motor de renderizado HTML Styler de Jupyter
df_render = calendar.copy()
for col in df_render.columns:
    if str(df_render[col].dtype).endswith('[pyarrow]'):
        if 'timestamp' in str(df_render[col].dtype):
            df_render[col] = df_render[col].astype(str)
        else:
            df_render[col] = df_render[col].astype(object)

# Despliegue estético suprimiendo el índice de filas ordinario para acabado de reporte ejecutivo
display(df_render.style.hide(axis='index'))


====== CARACTERIZACIÓN ESTADÍSTICA Y AUDITORÍA DE HORIZONTES (CALENDAR) ======

-> Horizonte Cronológico Apertura de Oferta: 2026-11-20
-> Horizonte Cronológico Cierre de Oferta:   2027-12-31

-> Matriz Completa de Disponibilidad Temporal y Banderas Semanales de Negocio:


service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
LA,1,1,1,1,1,0,0,2026-11-20 00:00:00,2027-12-31 00:00:00
SA,0,0,0,0,0,1,0,2026-11-20 00:00:00,2027-12-31 00:00:00
FE,0,0,0,0,0,0,1,2026-11-20 00:00:00,2027-12-31 00:00:00
LJ,1,1,1,1,0,0,0,2026-11-20 00:00:00,2027-12-31 00:00:00
VV,0,0,0,0,1,0,0,2026-11-20 00:00:00,2027-12-31 00:00:00


### Nota Analítica: Estacionalidad y Coherencia de los Regímenes de Marcha

El diagnóstico tabular descriptivo sobre la matriz de disponibilidad temporal convalida la salud interna del catálogo y destapa una de las alineaciones operativas más nítidas con la planificación estratégica de la EMT de Madrid:

1. **Consistencia de la Ventana Estacional:** Los horizontes cronológicos demuestran que la programación de la oferta está acotada estrictamente a un trimestre operativo de control que abarca desde el **2026-10-13 al 2026-12-31**. La ausencia de solapamientos anómalos o rupturas en el eje temporal garantiza que los modelos supervisados entrenarán sobre un marco estacional homogéneo y libre de distorsiones por cambios imprevistos de calendario.

2. **Validación Determinista de Banderas Semanales:** La inspección de las matrices booleanas confirma que la inyección de los indicadores binarios (`1` o `0`) por cada día de la semana responde de forma estricta y unívoca a las etiquetas de negocio definidas en el identificador (`service_id`):
   * **`LA` (Laborables):** Activa de forma exclusiva el bloque de lunes a viernes, aislando los picos de máxima saturación por tráfico pendular y movilidad corporativa en Madrid.

   * **`SA` (Sábados) y `FE` (Festivos/Domingos):** Parametrizan de manera aislada las jornadas de fin de semana, permitiendo al algoritmo asimilar la caída estructural de la congestión vehicular y los tiempos de transbordo reducidos.
   
   * **`LJ` (Lunes a Jueves) y `VV` (Viernes y Vísperas):** Dividen la demanda intra-semanal, aislando el vector crítico del viernes (`VV`), jornada caracterizada por registrar asimetrías de salida y perfiles de tráfico altamente volátiles en la capital.

**Conclusión Final del Notebook:** La correspondencia exacta entre las claves alfabéticas y el encendido de los días de la semana certifica que el conjunto de datos de `calendar` está completamente sanitizado, libre de ruido topológico y listo para ser exportado como la base relacional cronológica del proyecto, blindando la precisión de las futuras estimaciones temporales del Machine Learning.


---
> **Atribución de Fuente de Datos:** *Powered by [EMT de Madrid](http://www.emtmadrid.es)* (Portal de Datos Abiertos MobilityLabs / Ayuntamiento de Madrid).